### CS/ECE/ISyE 524 — Introduction to Optimization — Summer 2026 ###

# CitiBike Bike Rebalancing Optimization — Progress Report #

#### Junkai Zhang (jzhang3257@wisc.edu)
#### Zhenyu Gui (zgui6@wisc.edu)
#### Xiaonan Meng (xmeng67@wisc.edu)

### Table of Contents

1. [Introduction](#1-introduction)
2. [Mathematical Models](#2-mathematical-models)
3. [Work Completed](#3-work-completed)
4. [Work Remaining](#4-work-remaining)
5. [Issues & Concerns](#5-issues--concerns)

## 1. Introduction ##

CitiBike serves New York City, Jersey City, and Hoboken with over 200 stations. The core operational challenge is **bike rebalancing**: riders create imbalances — some stations overflow (surplus) while others run empty (deficit). Each night, an operator must deploy a fleet of trucks to reposition bikes from surplus to deficit stations, minimizing total travel distance while satisfying as much demand as possible.

This is a classic **minimum-cost network flow** problem with integer constraints — a natural application of the LP, MILP, and robust optimization techniques covered in ISyE 524. The problem belongs to the family of **Vehicle Routing Problems with Pickup and Delivery (VRPPD)** and was first formulated as a mathematical optimization problem by Raviv, Tzur, and Forma [1] in 2013.

**Data source**: CitiBike trip data for May 2026 (Jersey City), publicly available from [CitiBike System Data](https://citibikenyc.com/system-data). The raw dataset contains 94,995 trips across 224 stations. We aggregate net flow per station (arrivals − departures), classify stations as surplus (net > 0) or deficit (net < 0), and compute pairwise Haversine distances. The system has a structural supply shortage: 933 bikes available vs. 1,288 needed (355-bike gap).

**Key simplifying assumptions**:
- **Static single-period**: rebalancing occurs overnight; no time windows or traffic
- **Deterministic demand**: net flow computed from one month's data, treated as known
- **Haversine distances**: straight-line approximation (symmetric); road-network effects analyzed separately in Model 3
- **Single depot**: trucks start and end at a synthetic depot at the station centroid
- **Identical trucks**: all trucks have the same capacity Q = 50 bikes

We formulate four progressively more sophisticated models — from a simple LP transportation problem to a robust optimization under uncertainty — and validate each on 10, 25, and 50-station subsets before the full 224-station network.

[1] T. Raviv, M. Tzur, and I. A. Forma, "Static repositioning in a bike-sharing system: models and solution approaches," *EURO Journal on Transportation and Logistics*, 2(3), pp. 187–229, 2013.

## 2. Mathematical Models ##

We formulate four models of increasing sophistication. All models are implemented in Julia + JuMP with the HiGHS solver.

### Model 1 — Uncapacitated Transportation Problem (LP)

The simplest formulation: ignore trucks and treat surplus stations as pure sources and deficit stations as pure sinks. Decision variables $y_{ij} \ge 0$ represent bikes moved from surplus $i$ to deficit $j$.

$$\begin{aligned}
\underset{y}{\text{minimize}} \quad & \sum_{i \in S} \sum_{j \in D} d_{ij} \, y_{ij} \\
\text{subject to} \quad & \sum_{j \in D} y_{ij} \leq s_i \quad \forall i \in S \\
& \sum_{i \in S} y_{ij} = b_j \quad \forall j \in D \\
& y_{ij} \geq 0
\end{aligned}$$

**Total unimodularity**: The constraint matrix has at most one +1 and one −1 per column, guaranteeing integer optimal solutions without branching.

### Model 2 — Capacitated Vehicle Routing with MTZ (MILP)

Adds trucks with capacity $Q = 50$, depot start/end, and explicit route sequencing via MTZ subtour elimination. Key innovation: **arc sparsification** — each station connects only to its $M$ nearest neighbors plus the depot, reducing binary variables from $O(N^2 \cdot K)$ to $O(N \cdot M \cdot K)$ (a 92% reduction for the full dataset).

### Model 3 — Asymmetric Distance Sensitivity Analysis (LP)

Perturbs the symmetric Haversine matrix with random noise to simulate one-way streets and road geometry. Measures plan divergence, cost gap, and volume change between symmetric and asymmetric optimal plans.

### Model 4 — Robust Optimization under Uncertainty (LP)

Applies Bertsimas–Sim budgeted uncertainty [2] to protect against up to $\Gamma$ stations simultaneously realizing worst-case supply/demand deviations. The full dual derivation is provided in the final report.

[2] D. Bertsimas and M. Sim, "The price of robustness," *Operations Research*, 52(1), pp. 35–53, 2004.

## 3. Work Completed ##

### Data Processing
- Downloaded and cleaned CitiBike May 2026 trip data (94,995 trips, 224 stations)
- Aggregated net flow per station, computed Haversine distance matrix
- Generated 10, 25, and 50-station subsets for progressive validation

### Model Implementation

| Model | Status | Solver | Solve Time | Key Result |
|-------|--------|--------|------------|------------|
| Model 1 (LP) | ✅ Complete | HiGHS | <0.01s | 933 bikes, 200 arcs, 1.80 km avg |
| Model 2 (MILP) | ✅ Complete | HiGHS | 300s (10–50 stns) | Sparse: 8.6–32.0% gap; 287–414 bikes moved |
| Model 3 (Asym) | ✅ Complete | HiGHS | <0.01s | Haversine error ≤2% cost impact |
| Model 4 (Robust) | ✅ Complete | HiGHS | <0.01s | Infeasible for Γ≥1 (structural gap) |

### Model 2 Key Improvements
The original dense MILP formulation (304K binary variables) was computationally intractable. We implemented five improvements:
1. **Arc sparsification**: M-nearest neighbors per station, 92% variable reduction
2. **MTZ only on existing arcs**: avoids redundant constraints for unused trucks
3. **Tight big-M**: $\min(s_i, Q)$ instead of $s_i$ in pickup/dropoff linking
4. **Return empty**: trucks must return to depot empty
5. **Adaptive visit-once**: stations with supply/demand > Q allow multiple visits

### Results Summary (Model 2, Sparse Arc Formulation)

| Scale | M | Bin Vars | Trucks | Bikes | Unmet | Truck-km | Gap |
|-------|---|----------|--------|-------|-------|----------|-----|
| 10 stns | 5 | 140 | 2/2 | 287 | 289 | 31 | 8.6% |
| 25 stns | 8 | 750 | 3/3 | 414 | 452 | 49 | 16.1% |
| 50 stns | 10 | 2,400 | 4/4 | 338 | 734 | 59 | 32.0% |

### Final Report
- Draft complete with all four models, results, discussion, and conclusion
- Code and report are version-controlled on GitHub(a private link)

## 4. Work Remaining ##

We still have a few things to wrap up before the final report is due. Some of these are new tasks that came up after we dug into the model behavior more carefully.

**Model 2 tuning.** Here is the situation: our sparse version of Model 2 is actually slower than the original dense formulation for the 10-station case. The dense model solved in under a second, but our sparse version hits the 300-second time limit. That is not what we expected, and we spent some time figuring out why.

The main culprit is the adaptive visit-once constraint. We relaxed it so stations with high supply or demand can be visited more than once (ceil(max(supply, demand) / Q) times). That sounds reasonable in theory, but in practice it blows up the search space. With the relaxation, the solver has to consider many more routing combinations and the branch and bound tree gets a lot deeper. The fix is straightforward: go back to at most one visit per station and let the unmet demand variable handle the excess. The return_empty constraint is also making the problem tighter than it needs to be, and we might drop it so the solver has more flexibility. Separately, M=5 neighbors for 10 stations is too many. With only 9 other stations, connecting to 5 of them means the sparse graph is still more than half dense. Reducing M to 3 should help.

**Model 4 fix.** Right now Model 4 is infeasible for any Γ above zero because the robust demand constraint does not include the unmet demand variable. The constraint requires total flow to cover the nominal demand plus the robust buffer, but there is no mechanism to say "we tried our best but could not cover everything." If we add unmet to that constraint (with a penalty), the model will have feasible solutions for Γ>0 and we can actually plot a price of robustness curve. My teammate already flagged this with a TODO in src/model4_robust.jl.

**Visualizations.** The final report only has one figure right now, a histogram of net flows and a dot map of station locations. We need to add route maps showing the truck paths on actual Jersey City geography, a Pareto frontier for the cost versus unmet demand tradeoff, and sensitivity curves for truck capacity and fleet size. Plots.jl can handle all of these.

**Text polishing.** Some sections of the final report read a bit stiff and we want to go through them to make the writing more natural before submitting.

**Out of sample validation.** If we have time, we want to test the models on July 2026 trip data to see how well the June based plans hold up. This is a nice to have, not a must.

**Time estimates:**

| Task | Hours | Priority |
|------|-------|----------|
| Model 2 tuning and rerunning all scales | 3 | High |
| Model 4 unmet fix and rerun | 1 | High |
| Visualizations (routes, Pareto, sensitivity) | 2–3 | Medium |
| Text polishing | 2 | High |
| Out of sample validation (July data) | 3 | Low |
| Export to PDF | 0.5 | High |

The remaining work should take about 8 to 10 hours total. Model 2 tuning is the biggest chunk. The Model 4 fix is quick, maybe an hour. We plan to finish the high priority items this week.

## 5. Issues & Concerns ##

We have two main issues right now, and both are fixable.

**Issue 1: Model 2 is slower than expected.** I noted this in the work remaining section, but the short version is: the adaptive visit-once constraint is the main problem. The dense model solved 10 stations in 0.8 seconds and hit optimality. Our sparse model hits the 300 second time limit with an 8.6% gap. We know what is causing it and we know how to fix it. The concern is that even after fixing the visit-once constraint, the MTZ formulation's LP relaxation is known to be weak, so the gap will still grow with problem size. But we should at least get back to sub-second solve times for the small cases, and the gap should come down noticeably.

We also need to be more careful about how we choose M, the number of neighbors per station. Right now we use fixed values per dataset size (5, 8, 10, 15), but the optimal M depends on the spatial distribution of the stations. For the full 224 station dataset, M=15 is probably fine. For the 10 station subset, M=3 makes more sense than M=5. A quick test with M=3 should confirm this.

**Issue 2: Model 4 cannot produce a price of robustness curve.** The model goes infeasible as soon as Γ hits 1, so we cannot show how cost increases with the uncertainty budget. The root cause is clear: the robust demand constraint (Σy ≥ Σb̄ + Γ·z_d + Σt_d) does not include the unmet demand variable. The model has no way to say "some demand is not covered" within the robust constraint, so it gives up and declares infeasibility. Adding unmet to that constraint is a one line change. Once that is done, the model will have feasible solutions for Γ>0 and we can sweep Γ to plot the actual price of robustness. I am not worried about this one.

One thing we are no longer concerned about is the Haversine distance approximation. Model 3 showed that even with 20% random perturbations, the total cost only changes by about 2%. For the scale of this project and the precision we need, straight line distances are fine.

Overall, the project is in decent shape. The four models are all implemented and running. The computational bottleneck with Model 2 is real but we understand the root cause and have a clear fix. The Model 4 issue is trivial to fix. The main remaining work is polishing the report and making the visualizations look good. We do not anticipate any blocking issues.